# Lab 7: Model Evaluation and Cross-Validation

**Course**: Machine Learning — Undergraduate  
**Dataset**: Olist Brazilian E-Commerce Dataset (Feature Engineered)  
**Instructor**: Sharad Laad | ORY AI Labs  

---

## 1. Lab Overview & Learning Objectives
- Explain why model evaluation beyond raw accuracy is mandatory for imbalanced datasets.
- Distinguish between training, validation, and test datasets.
- Construct and interpret a Confusion Matrix (TP, TN, FP, FN).
- Calculate and interpret Accuracy, Precision, Recall, F1-Score, and ROC-AUC.
- Explore classification thresholds and the precision-recall trade-off.
- Apply $k$-fold cross-validation (`StratifiedKFold`) to quantify model generalization and fold stability (mean and standard deviation).
- Identify the 6 critical evaluation mistakes and implement a leak-free `Pipeline` evaluation.

## 2. Part A & B: Load Dataset, Check Class Imbalance, and Define X & y

Business Question: **Will this order be delivered late?** (`is_late_delivery`)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

input_path = Path("data/processed/olist_orders_feature_engineered.csv")
if not input_path.exists():
    input_path = Path("../lab06/data/processed/olist_orders_feature_engineered.csv")

df = pd.read_csv(input_path)
print("Dataset Shape:", df.shape)

target = "is_late_delivery"
print("\nTarget Class Counts:")
print(df[target].value_counts())

print("\nTarget Proportions (%):")
print(df[target].value_counts(normalize=True) * 100)

# Define X and y
y = df[target]
X = df.drop(columns=[target]).select_dtypes(include=np.number)
X = X.fillna(X.median())

print(f"\nFeatures shape X: {X.shape}, Target shape y: {y.shape}")

## 3. Part C & D: Train-Test Split and Baseline Model Training

We perform an 80/20 stratified split (`stratify=y`) to maintain the minority class proportion in both training and test sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

# Train baseline model
model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print("Model trained successfully!")

## 4. Part E to K: Confusion Matrix, Accuracy, Precision, Recall, F1, and ROC-AUC

A confusion matrix summarizes classification predictions:
- **True Positive (TP)**: Model predicted Late, and the order was actually Late.
- **True Negative (TN)**: Model predicted On-time, and the order was actually On-time.
- **False Positive (FP)**: Model predicted Late, but the order was On-time (false alarm).
- **False Negative (FN)**: Model predicted On-time, but the order was Late (missed delivery delay).

In [ ]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, classification_report
)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print(f"TN: {cm[0, 0]}, FP: {cm[0, 1]}")
print(f"FN: {cm[1, 0]}, TP: {cm[1, 1]}")

print("\nSingle Test Set Metrics:")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 5. Section 17 & 29: Classification Thresholds Analysis

The default decision boundary is $P(\text{late}) \ge 0.50$. In business problems with asymmetric costs, shifting the threshold balances False Positives vs. False Negatives.

In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.70]
for th in thresholds:
    pred_th = (y_prob >= th).astype(int)
    p = precision_score(y_test, pred_th, zero_division=0)
    r = recall_score(y_test, pred_th, zero_division=0)
    f = f1_score(y_test, pred_th, zero_division=0)
    print(f"Threshold {th:.2f} -> Precision: {p:.4f} | Recall: {r:.4f} | F1: {f:.4f}")

## 6. Part M, N, O: 5-Fold Stratified Cross-Validation & Metric Stability

Instead of trusting a single train/test split, $k$-fold cross-validation trains and validates across 5 non-overlapping folds to determine mean performance and standard deviation.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")

print("5-Fold F1 Scores:", [round(s, 4) for s in cv_scores])
print(f"Mean F1: {cv_scores.mean():.4f} | Std F1: {cv_scores.std():.4f}")

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
cv_multi = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
print("\nMulti-Metric CV Summary:")
for m in scoring:
    vals = cv_multi["test_" + m]
    print(f"{m:<12}: Mean = {vals.mean():.4f}, Std = {vals.std():.4f}")

## 7. Part Q & Section 26: Leak-Free Pipeline Cross-Validation and Final Test Evaluation

To eliminate preprocessing data leakage, `SimpleImputer` and `StandardScaler` are wrapped with `LogisticRegression` inside a `Pipeline`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

# CV evaluation inside pipeline
pipe_cv = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1")
print(f"Pipeline 5-Fold CV F1: Mean = {pipe_cv.mean():.4f}, Std = {pipe_cv.std():.4f}")

# Final evaluation on untouched test set
pipeline.fit(X_train, y_train)
final_pred = pipeline.predict(X_test)
final_prob = pipeline.predict_proba(X_test)[:, 1]

print("\nFinal Test Set Results:")
print(f"Accuracy : {accuracy_score(y_test, final_pred):.4f}")
print(f"Precision: {precision_score(y_test, final_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, final_pred, zero_division=0):.4f}")
print(f"F1-Score : {f1_score(y_test, final_pred, zero_division=0):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, final_prob):.4f}")